# Chapter 2: Getting Used to Errors Everywhere

This notebook accompanies **Chapter 2** of the lecture notes.

**Agenda**

🔬 · 🔢 · 💥 · 📈 · 🧮 · 🏁

**Next steps (take it from here):** 🔍 · ⚖️

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones.

> **Scenario:** Your coffee lab just received a new digital scale for weighing green coffee beans before roasting. The display shows weight to a certain number of decimal places — but how much can you trust those digits?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from checks import (check_machine_epsilon, check_cancellation,
                    check_safe_subtract)


def tufte_axis(ax):
    """Remove spines, keep only outward ticks on left and bottom."""
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(axis='both', which='both', direction='out',
                   length=5, width=1.2, colors='black',
                   top=False, right=False)

## Your Coffee Scale Lies

Every digital measurement your coffee lab's scale performs carries a tiny amount of error. Most of the time this does not matter — whether a dose is 18.0000 g or 18.0001 g makes no difference to the cup. But when you compute small differences between nearly equal weights (e.g., tracking moisture loss during roasting), the results can be wildly wrong. This chapter is about learning to see those errors and work around them.

> Before running the cell below, predict: does `0.1 + 0.2 == 0.3` evaluate to `True` or `False` in Python?

<details><summary>Thought</summary>

It evaluates to `False`. The decimal fractions 0.1 and 0.2 cannot be represented exactly in binary floating point. Their stored approximations sum to something like 0.30000000000000004, which is not bitwise equal to the stored approximation of 0.3. This is not a Python bug — it is a fundamental property of IEEE 754 arithmetic. Your scale's microprocessor faces the same limitation.
</details>

In [ ]:
print("0.1 + 0.2 == 0.3?", 0.1 + 0.2 == 0.3)
print("0.1 + 0.2       =", repr(0.1 + 0.2))

### 🔬 Machine Epsilon

> Think of your coffee scale's display precision. A cheap scale shows whole grams, a good one shows 0.1 g, and a lab-grade one shows 0.01 g. Floating-point numbers have an analogous resolution limit — an epsilon below which adding a small value to 1.0 is indistinguishable from 1.0 itself. If you halve epsilon repeatedly, at what point does the computer stop noticing the difference, and why does that threshold depend on the data type?

<details><summary>Thought</summary>

Floating-point numbers have a fixed number of significand bits. `float16` has 10 fraction bits, `float32` has 23, and `float64` has 52. Once epsilon is small enough that `1 + eps` rounds back to `1` within those bits, the addition is absorbed. More bits means a smaller epsilon and finer resolution — like upgrading from a kitchen scale to an analytical balance.
</details>

Write a function that finds the machine epsilon for a given NumPy dtype by halving. Start with `eps = 1` and keep halving while `dtype(1) + dtype(eps) > dtype(1)`. Return the last epsilon that was still distinguishable.

Useful operations: `dtype()` to cast a value, simple arithmetic (`/`, `>`).

In [ ]:
def find_machine_epsilon(dtype):
    """Return the machine epsilon for the given NumPy float dtype."""
    eps = dtype(1)
    while dtype(1) + dtype(eps) > dtype(1):
        last = eps
        eps = dtype(eps / 2)
    return float(last)


check_machine_epsilon(find_machine_epsilon, np.float32)

### 🔢 Epsilon Across Data Types

> Half-precision, single-precision, and double-precision formats allocate 10, 23, and 52 fraction bits respectively. Each extra bit roughly halves the machine epsilon. Run your function on all three and compare to NumPy's built-in `np.finfo`.

Run the cell below to see the results side by side.

In [ ]:
for dt in [np.float16, np.float32, np.float64]:
    check_machine_epsilon(find_machine_epsilon, dt)

print()
print("NumPy reference values:")
for dt in [np.float16, np.float32, np.float64]:
    print(f"  np.finfo({dt.__name__}).eps = {np.finfo(dt).eps:.6e}")

### 💥 Cancellation Error

> Imagine weighing a batch of green beans (say 500.000 g) before and after roasting. The beans lose only a tiny amount of moisture — perhaps 0.001 g in a short test. If your scale stores both weights as floating-point numbers and you subtract them, the leading digits cancel and the result is dominated by rounding noise. Suppose `a = 1.0` and `b = 1.0`, and you want to compute `(a + eps) - b` for a tiny `eps`. Mathematically the answer is just `eps`. At what point does the computer give a different answer, and why?

<details><summary>Thought</summary>

When `eps` is small enough, `a + eps` rounds to `a` (because of machine epsilon). Then `(a + eps) - b` becomes `a - b = 0` instead of `eps`. Even before total loss, the intermediate result `a + eps` discards the low-order bits of `eps` during the addition, so the subtraction of `b` amplifies those rounding errors. This is catastrophic cancellation — the same thing that happens when your scale tries to measure a tiny weight change on top of a large tare.
</details>

Write a function that takes `a`, `b`, and `eps`, and returns a tuple: the naive computed result `(a + eps) - b` and the mathematically exact result `eps`.

Useful operations: basic arithmetic (`+`, `-`), return a tuple `(x, y)`.

In [ ]:
def cancellation_error(a, b, eps):
    """Return (naive_result, true_result) showing cancellation."""
    naive_result = (a + eps) - b
    true_result = eps
    return (naive_result, true_result)


check_cancellation(cancellation_error, 1.0, 1.0, 1e-15)

### 📈 Visualizing Cancellation

> The plot below sweeps epsilon from `1e-1` down to `1e-18` and shows the absolute error between the naive computation `(1 + eps) - 1` and the true value `eps`. Think of epsilon as the tiny moisture loss you are trying to measure. At large epsilon the error is near zero. Where does it start to grow, and how does that relate to `float64` machine epsilon?

Run the cell to generate the plot. If you have not yet implemented `cancellation_error`, do that first.

In [ ]:
epsilons = np.logspace(-1, -18, 200)
errors = []

for eps in epsilons:
    result = cancellation_error(1.0, 1.0, eps)
    if result is None:
        errors.append(np.nan)
    else:
        naive, true = result
        errors.append(abs(naive - true))

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(epsilons, errors, 'k-', linewidth=0.8)
ax.axvline(np.finfo(np.float64).eps, color='tab:red', linewidth=0.8,
           linestyle='--', label=f'float64 eps = {np.finfo(np.float64).eps:.1e}')
ax.set_xlabel('epsilon')
ax.set_ylabel('absolute error |naive - true|')
ax.legend(frameon=False, fontsize=9)
tufte_axis(ax)
plt.tight_layout()
plt.show()

**Observe:**
- For large epsilon (left side), the error is negligible — floating point handles the addition just fine.
- As epsilon approaches and passes the machine epsilon (red dashed line), the error grows rapidly.
- Below machine epsilon, `(1 + eps) - 1` rounds to exactly 0, and the absolute error equals `eps` itself — total loss of significance.

### 🧮 Safe Subtraction

> Back in the coffee lab: instead of weighing the full batch, adding a tiny correction, then subtracting the original weight, you could weigh the moisture loss directly (tare the scale with the beans on it, then measure the change). The same idea applies to arithmetic. The naive formula `(a + eps) - b` fails because adding a tiny `eps` to a large `a` loses precision before the subtraction ever happens. Can you rearrange the same three terms so that the small quantity is never absorbed into a large one?

<details><summary>Thought</summary>

Rewrite `(a + eps) - b` as `eps + (a - b)`. When `a` and `b` are equal (or nearly equal), `a - b` is computed exactly (or very accurately) because they are close in magnitude. Then adding `eps` to that small difference preserves all the precision of `eps`. The key insight is: subtract the large quantities from each other first, then add the small correction.
</details>

Write a function that computes `(a + eps) - b` in a numerically safe way by rearranging the terms.

Useful operations: basic arithmetic (`+`, `-`). The trick is the order of operations.

In [ ]:
def safe_subtract(a, eps, b):
    """Compute (a + eps) - b without cancellation."""
    return eps + (a - b)


check_safe_subtract(safe_subtract, 1.0, 1e-15, 1.0)

### 🏁 Recap

**What we did:**
- 🔬 Found machine epsilon by halving — the resolution limit of floating-point arithmetic, analogous to the smallest increment on your coffee scale.
- 🔢 Compared epsilon across `float16`, `float32`, and `float64` and matched our results to `np.finfo`.
- 💥 Demonstrated catastrophic cancellation when subtracting nearly equal numbers — the same problem you hit when computing tiny weight differences from large measurements.
- 📈 Visualized how the error grows as epsilon shrinks past the machine epsilon threshold.
- 🧮 Rearranged a subtraction to avoid cancellation — order of operations matters, just like taring your scale before measuring a small change.

**Key takeaways:**
1. Floating-point arithmetic is approximate. Every operation can introduce a small rounding error.
2. Catastrophic cancellation happens when nearly equal numbers are subtracted, amplifying rounding noise.
3. Rearranging mathematically equivalent expressions can dramatically improve numerical accuracy.

**Now head back for self-check questions and key learnings in the lecture notes.**

## Take It from Here — Next Steps (Optional)

The exercises below are **optional** extensions. They deepen your intuition but are not required to follow the rest of the course. Work through them at your own pace after the session.

### 🔍 Integer Precision

> When your coffee lab's inventory system counts whole bags of beans, it uses integers — exact, no rounding. But integers have a different limitation: a maximum value. What happens when you exceed it?

<details><summary>Thought</summary>

NumPy integer types wrap around silently on overflow. Adding 1 to the maximum `int64` value gives the minimum (most negative) `int64` value. Python's built-in `int` has arbitrary precision and never overflows, but NumPy arrays use fixed-width types for performance. This is a different kind of "error everywhere" — not rounding, but overflow.
</details>

Run the cells below to see integer overflow in action.

In [ ]:
max_int64 = np.iinfo(np.int64).max
print(f"Max int64: {max_int64}")
print(f"Max int64 + 1 (NumPy): {np.int64(max_int64) + np.int64(1)}")
print(f"Max int64 + 1 (Python): {max_int64 + 1}")
print()
print("NumPy wraps around silently. Python integers have arbitrary precision.")
print("This is why numerical code must be aware of its data types.")

### ⚖️ Safe vs. Naive — Range Comparison

> How much does the rearrangement actually help across the full range of epsilon values? Let's plot the naive and safe results side by side.

Run the cell below to compare `cancellation_error` and `safe_subtract` across the same epsilon sweep.

In [ ]:
epsilons = np.logspace(-1, -18, 200)
naive_results = []
safe_results = []

a, b = 1.0, 1.0
for eps in epsilons:
    result = cancellation_error(a, b, eps)
    if result is not None:
        naive_results.append(result[0])
    else:
        naive_results.append(np.nan)

    safe_val = safe_subtract(a, eps, b)
    if safe_val is not None:
        safe_results.append(safe_val)
    else:
        safe_results.append(np.nan)

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(epsilons, epsilons, 'k--', linewidth=0.6, label='true value (eps)')
ax.loglog(epsilons, np.abs(naive_results), 'tab:red', linewidth=0.8,
          label='naive: (a + eps) - b')
ax.loglog(epsilons, np.abs(safe_results), 'tab:blue', linewidth=0.8,
          label='safe: eps + (a - b)')
ax.axvline(np.finfo(np.float64).eps, color='gray', linewidth=0.6,
           linestyle=':', label=f'float64 eps')
ax.set_xlabel('epsilon')
ax.set_ylabel('computed result')
ax.legend(frameon=False, fontsize=9)
tufte_axis(ax)
plt.tight_layout()
plt.show()

**Observe:**
- The safe formula (blue) tracks the true value (dashed) perfectly across the entire range.
- The naive formula (red) diverges from the true value as epsilon approaches machine epsilon, and drops to zero below it.
- A simple reordering of three arithmetic operations makes the difference between a correct answer and total loss of information.